In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


class MainState(TypedDict):
    user_input: str
    route: str
    final_answer: str


# =========================
# Main Router
# =========================

def router_node(state: MainState) -> MainState:
    prompt = f"""
사용자 질문을 보고 어떤 작업인지 분류해라.

분류 기준:
- DB 조회, 매출, 상품, 지점, 통계 질문이면 text_to_sql
- 문서 검색, 규정, 보고서 질문이면 rag
- 이미지 생성, 편집이면 image
- 그 외는 general

사용자 질문:
{state["user_input"]}

분류명만 출력:
"""

    response = llm.invoke(prompt)
    route = response.content.strip().lower()

    return {
        **state,
        "route": route
    }


def route_main(
    state: MainState
) -> Literal["text_to_sql", "rag", "image", "general"]:
    route = state["route"]

    if "text_to_sql" in route:
        return "text_to_sql"

    if "rag" in route:
        return "rag"

    if "image" in route:
        return "image"

    return "general"


# =========================
# A. Text-to-SQL Subgraph 호출
# =========================

def text_to_sql_node(state: MainState) -> MainState:
    result = text_to_sql_graph.invoke({
        "question": state["user_input"],
        "schema": "",
        "sql": "",
        "result": "",
        "error": "",
        "retry_count": 0,
        "answer": ""
    })

    return {
        **state,
        "final_answer": result["answer"]
    }


# =========================
# B, C, D 임시 노드
# =========================

def rag_node(state: MainState) -> MainState:
    return {
        **state,
        "final_answer": "RAG 문서 검색 Subgraph로 연결하면 됩니다."
    }


def image_node(state: MainState) -> MainState:
    return {
        **state,
        "final_answer": "ComfyUI 이미지 생성 Subgraph로 연결하면 됩니다."
    }


def general_node(state: MainState) -> MainState:
    response = llm.invoke(state["user_input"])

    return {
        **state,
        "final_answer": response.content
    }


# =========================
# Main Graph 생성
# =========================

main_builder = StateGraph(MainState)

main_builder.add_node("router", router_node)
main_builder.add_node("text_to_sql", text_to_sql_node)
main_builder.add_node("rag", rag_node)
main_builder.add_node("image", image_node)
main_builder.add_node("general", general_node)

main_builder.add_edge(START, "router")

main_builder.add_conditional_edges(
    "router",
    route_main,
    {
        "text_to_sql": "text_to_sql",
        "rag": "rag",
        "image": "image",
        "general": "general"
    }
)

main_builder.add_edge("text_to_sql", END)
main_builder.add_edge("rag", END)
main_builder.add_edge("image", END)
main_builder.add_edge("general", END)

main_graph = main_builder.compile()